# 02 · Power score sensitivity to the weights

How much do the power rankings depend on the exact weights in `config.yaml`? Each weight is nudged by ±25%, one at a time, and the resulting ranking is compared with the baseline. Nothing here changes any data or settings.

To see the latest numbers, run `python -m sleeper_dash.pipeline` first, then **Run → Restart Kernel and Run All Cells**. The written interpretations below describe the run **as of week 3 (2026-10-02)**; the summary cell near the end regenerates its sentences from whatever data you load.

## Method

**Baseline.** The power score exactly as the pipeline computes it (`metrics/power.py`, `METRICS_SPEC.md` section 6), with the weights from `config.yaml`.

**Perturbations.** For each of the four components *c*, the weight is multiplied by 0.75 (−25%) and by 1.25 (+25%). The weights must still sum to 1, so the other three are **rescaled proportionally** to absorb the difference: *w′₍c₎* = 1.25 · *w₍c₎*, and every other *w′₍k₎* = *w₍k₎* · (1 − *w′₍c₎*) ÷ (1 − *w₍c₎*). This keeps the other components' weights in the same proportion to each other, so each scenario changes exactly one thing: how much component *c* counts relative to the rest. That gives 8 scenarios.

**What is compared,** for the latest completed week:
- **Teams that change rank** versus the baseline.
- **Spearman rank correlation** with the baseline ranking. With 12 teams, one swap of neighbours gives ρ = 0.993, two swaps 0.986, and a team moving two places (a three-way rotation) 0.979. So ρ values here are best read through those benchmarks; anything above 0.95 means the order barely moved.
- **Largest move**: the most places any one team moves.
- As a cross-check, the same count of rank changes over **every week so far**.

Rankings for every scenario are recomputed with the pipeline's own function, so they include every rule in the spec (early-season factor, tie-breaks).

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import spearmanr

from sleeper_dash.config import load_config
from sleeper_dash.metrics.power import COMPONENTS, build_power_rankings
from sleeper_dash.validate import load_tables

pd.set_option("display.width", 200)
config = load_config()
base_params = config.metrics["power"]
from sleeper_dash.seasons import season_slice

tables = season_slice(load_tables(), config.season)  # the current season only
names = tables["teams"].set_index("roster_id")["team_name"]
team_weeks, lineups = tables["team_weeks"], tables["lineups_optimal"]

baseline_all = build_power_rankings(team_weeks, lineups, base_params)
week = int(baseline_all["week"].max())
baseline = baseline_all[baseline_all["week"] == week].set_index("roster_id")
print(f"Latest completed week: {week}. Baseline weights: " + ", ".join(f"{c} {base_params['weights'][c]:.2f}" for c in COMPONENTS))
baseline.sort_values("rank").assign(team=lambda d: d.index.map(names))[["rank", "team", "power_score"]].round(3)

## Run the eight scenarios

The table shows each scenario's rescaled weights (in the order season scoring, recent form, roster strength, results), how many teams changed rank in the latest week, the Spearman correlation with the baseline, the largest single move, and the number of team-week ranks that changed across every week so far.

In [ ]:
def perturb(weights, component, factor):
    """Scale one weight by `factor`; rescale the others proportionally so the weights still sum to 1."""
    new = {component: weights[component] * factor}
    others = (1 - new[component]) / (1 - weights[component])
    new.update({c: w * others for c, w in weights.items() if c != component})
    return {c: new[c] for c in COMPONENTS}


rows, ranks, scores = [], {"baseline": baseline["rank"]}, {}
for component in COMPONENTS:
    for label, factor in (("−25%", 0.75), ("+25%", 1.25)):
        weights = perturb(base_params["weights"], component, factor)
        result = build_power_rankings(team_weeks, lineups, {**base_params, "weights": weights})
        latest = result[result["week"] == week].set_index("roster_id").reindex(baseline.index)
        every_week = result.merge(baseline_all, on=["season", "week", "roster_id"], suffixes=("", "_base"))
        name = f"{component} {label}"
        ranks[name], scores[name] = latest["rank"], latest["power_score"]
        rows.append({
            "scenario": name,
            "weights": " / ".join(f"{weights[c]:.3f}" for c in COMPONENTS),
            "teams_changed": int((latest["rank"] != baseline["rank"]).sum()),
            "spearman_rho": spearmanr(baseline["rank"], latest["rank"]).statistic,
            "largest_move": int((latest["rank"] - baseline["rank"]).abs().max()),
            "max_power_change": (latest["power_score"] - baseline["power_score"]).abs().max(),
            "rank_changes_all_weeks": int((every_week["rank"] != every_week["rank_base"]).sum()),
        })

scenarios = pd.DataFrame(rows)
scenarios.round({"spearman_rho": 3, "max_power_change": 2})

### What the scenario table shows (as of week 3)

- **The ranking is robust to the weights.** Spearman ρ is **0.979 or higher** in every scenario. Six of the eight scenarios swap exactly one pair of neighbouring teams (2 teams change, ρ = 0.993). The other two (season scoring −25% and results +25%) move one team two places (3 teams change, ρ = 0.979).
- **No team moves more than two places,** and only the top three and the 7th/8th pair ever move.
- **Power scores barely move.** The largest change to any team's score is 0.24–0.69 points depending on the scenario, while the gaps between neighbours outside the near-ties are 1.2–4.2 points. Weight changes of this size can only reorder teams that are already nearly level.
- **Through week 3, recent form *is* season scoring** (the 3-week window covers the whole season), so the two act as one scoring block weighted 0.60. The moves follow from that:
  - **Top three.** The two scenarios that shift weight from scoring toward results (season scoring −25%, results +25%) drop QB Whore No More, the top scorer at 2–1 head-to-head, to 3rd behind the two 3–0 teams. Two more (recent form −25%, roster strength −25%) just swap 1st and 2nd: with a 0.004-point gap, almost any change tips it.
  - **7th and 8th.** Downs bad (above-average scoring, 0–3) passes Tight End University in the three scenarios that reduce results' share relative to scoring (season scoring +25%, recent form +25%, results −25%) and when roster strength counts more (+25%), where Downs bad is also slightly ahead.
- **Across every week so far,** 4 to 8 of the 36 team-week ranks change per scenario, consistent with the latest week alone.

## Which teams are sensitive, and why

For each team: its baseline rank, its rank in each scenario, the best and worst rank across all scenarios, how many of the 8 scenarios moved it, and the **gap in power score** to the team directly above and below it in the baseline.

In [ ]:
rank_table = pd.DataFrame(ranks)
scenario_cols = [c for c in rank_table.columns if c != "baseline"]
ordered = baseline.sort_values("rank")["power_score"]
sensitivity = pd.DataFrame({
    "team": rank_table.index.map(names),
    "baseline_rank": rank_table["baseline"],
    "best_rank": rank_table.min(axis=1),
    "worst_rank": rank_table.max(axis=1),
    "scenarios_moved": rank_table[scenario_cols].ne(rank_table["baseline"], axis=0).sum(axis=1),
    "gap_to_team_above": (ordered.shift(1) - ordered).reindex(rank_table.index),
    "gap_to_team_below": (ordered - ordered.shift(-1)).reindex(rank_table.index),
}, index=rank_table.index).sort_values("baseline_rank")
sensitivity.round(3)

In [ ]:
order = sensitivity.sort_values("baseline_rank")
fig = go.Figure()
for r in order.itertuples():
    fig.add_trace(go.Scatter(x=[r.best_rank - 0.15, r.worst_rank + 0.15], y=[r.team, r.team], mode="lines",
                             line=dict(width=10, color="#c3ccd6"), hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=order["baseline_rank"], y=order["team"], mode="markers", name="Baseline rank",
                         marker=dict(size=11, color="#1f6f4a")))
fig.update_layout(title="Rank range across the 8 weight scenarios (grey bar) and baseline rank (dot)",
                  xaxis=dict(title="Rank (1 = best)", dtick=1, range=[0.5, 12.5]),
                  yaxis=dict(autorange="reversed", title=None), height=450, margin=dict(l=10, r=10, t=50, b=40))
fig.show()

In [ ]:
rank_table.assign(team=rank_table.index.map(names)).set_index("team").sort_values("baseline")

### What the team table shows (as of week 3)

- **Only five teams ever move, and each sits in a near-tie.**
  - **QB Whore No More (1st) and Lawrence & Order: SNU (2nd)** are separated by **0.004 points**. Any change in the weights flips them; QB Whore No More moves in 4 of 8 scenarios and drops as low as 3rd. **I have Mommy Issues (3rd)** is only 0.35 behind and rises to 2nd in two scenarios.
  - **Tight End University (7th) and Downs bad (8th)** are separated by **0.038 points** and swap in 4 of 8 scenarios.
- **The other seven teams never move.** Their gaps to their neighbours are 1.2–4.2 points, far larger than any change the ±25% scenarios can produce.
- **Why these particular pairs flip:** the teams in each pair are good at different things. QB Whore No More leads on scoring but is 2–1 head-to-head; Lawrence & Order and I have Mommy Issues score slightly less but are 3–0. Downs bad outscores Tight End University but is 0–3. When two teams have nearly equal totals built from different components, the weights decide the order.

A useful way to see this: when weight *w₍c₎* changes by Δ and the others are rescaled proportionally, team *i*'s power score changes by

  Δpower₍ᵢ₎ = scale · *f* · Δ · ( *z₍ᵢ,c₎* − z̄₍ᵢ,−c₎ ),

where z̄₍ᵢ,−c₎ is the weighted average of the team's z-scores on the *other* components. Two teams swap when the difference between their Δpower values exceeds the gap between them. Teams with similar component profiles move together; teams with opposite profiles (strong scoring and weak results, or the reverse) move apart.

## Plain-language summary of this run

This cell writes its sentences from the data you loaded, so it stays accurate after later weeks are added.

In [ ]:
lowest = scenarios.loc[scenarios["spearman_rho"].idxmin()]
moved = sensitivity[sensitivity["scenarios_moved"] > 0].sort_values("scenarios_moved", ascending=False)
stable = sensitivity[sensitivity["scenarios_moved"] == 0]
close = sensitivity[sensitivity["gap_to_team_below"] < 0.5].sort_values("baseline_rank")
print(f"As of week {week}:")
print(f"- Across the 8 scenarios, {scenarios['teams_changed'].min()}–{scenarios['teams_changed'].max()} of 12 teams change rank; "
      f"Spearman rho ranges from {scenarios['spearman_rho'].min():.3f} to {scenarios['spearman_rho'].max():.3f} "
      f"(lowest: {lowest['scenario']}).")
print(f"- No team moves more than {scenarios['largest_move'].max()} place(s); the largest change to any power score is "
      f"{scenarios['max_power_change'].max():.2f} points.")
print(f"- {len(stable)} teams never move. The teams that do: " + "; ".join(
    f"{r.team} (rank {r.baseline_rank}, moved in {r.scenarios_moved}/8, range {r.best_rank}–{r.worst_rank})" for r in moved.itertuples()) + ".")
print("- Near-ties in the baseline (gap to the next team under 0.5 points): " + ("; ".join(
    f"#{r.baseline_rank} {r.team} over #{r.baseline_rank + 1} by {r.gap_to_team_below:.3f}" for r in close.itertuples()) or "none") + ".")

## Caveats and what this means

- **Three weeks of data.** Every component is an average of at most three numbers, and recent form is identical to season scoring until week 4. The analysis is worth re-running mid-season, when the components have separated and the early-season factor no longer compresses the scores (the factor itself never changes the order, only the spread).
- **This tests the weights, not the components.** A ±25% change is a local check around the chosen weights. It says nothing about whether, say, roster strength belongs in the score at all. That question is better answered by the Phase 5 backtest idea in the spec (which weighting best predicts next week's results).
- **Conclusion for now:** the ranking is driven by real differences between teams, not by the precise weights. The only teams whose order depends on the weights are ones that are effectively tied, and for those the weights act as a tie-breaker between "scores more" and "wins more". If that matters on the dashboard, the place to handle it is the display (for example, showing near-equal scores to two decimals or marking them as tied), not the weights.